# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [17]:
# Core libraries and reproducibility
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

## Loading Data

We begin by loading the interaction data.

In [18]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [19]:
from sklearn.model_selection import train_test_split

# Map ids to a compact 0 to N-1 range
user2id = {u: i for i, u in enumerate(df['user_id'].unique())}
item2id = {i: j for j, i in enumerate(df['item_id'].unique())}

df['user_id'] = df['user_id'].map(user2id)
df['item_id'] = df['item_id'].map(item2id)

# Train/test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [20]:
# Build labelled data with sampled negatives
def sample_negatives(df, num_neg=1):
    all_items = df['item_id'].unique()

    # Precompute candidate negatives per user
    user_to_items = df.groupby('user_id')['item_id'].apply(set).to_dict()
    user_neg_pool = {
        u: np.array(list(set(all_items) - items))
        for u, items in user_to_items.items()
    }

    rows = []
    for _, row in df.iterrows():
        u = row['user_id']
        i = row['item_id']

        # Positive interaction
        rows.append((int(u), int(i), 1))

        # Sample negatives not in the user's history
        neg_pool = user_neg_pool.get(u, np.array([]))
        if len(neg_pool) == 0:
            continue

        neg_items = np.random.choice(
            neg_pool,
            size=num_neg,
            replace=len(neg_pool) < num_neg,
        )
        for neg_item in neg_items:
            rows.append((int(u), int(neg_item), 0))

    return pd.DataFrame(rows, columns=['user', 'item', 'label'])

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [21]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user'].values)
        self.items = torch.tensor(df['item'].values)
        self.labels = torch.tensor(df['label'].values).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [22]:
# Sample negatives for training and build a loader
train_data = sample_negatives(train_df, num_neg=4)
train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [23]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [24]:
# Matrix factorization model with user/item bias
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()  # init nn.Module
        self.user_emb = nn.Embedding(num_users, emb_dim)  # user embeddings
        self.item_emb = nn.Embedding(num_items, emb_dim)  # item embeddings
        self.user_bias = nn.Embedding(num_users, 1)  # user bias term
        self.item_bias = nn.Embedding(num_items, 1)  # item bias term
        nn.init.normal_(self.user_emb.weight, std=0.01)  # small init
        nn.init.normal_(self.item_emb.weight, std=0.01)  # small init
        nn.init.zeros_(self.user_bias.weight)  # zero bias init
        nn.init.zeros_(self.item_bias.weight)  # zero bias init

    def forward(self, user, item):  # forward pass
        user_vec = self.user_emb(user.long())  # user vectors
        item_vec = self.item_emb(item.long())  # item vectors
        scores = (user_vec * item_vec).sum(dim=1)  # dot product
        scores += self.user_bias(user).squeeze() + self.item_bias(item).squeeze()  # add biases
        return scores  # logits

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [ ]:
# Train with either a dataframe or a prebuilt dataloader
def train(model, train_source, epochs, lr=1e-3, num_neg=4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()

    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        if isinstance(train_source, DataLoader):
            train_loader = train_source
        else:
            train_data = sample_negatives(train_source, num_neg=num_neg)
            train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

        for user, item, label in train_loader:
            user = user.to(device).long()
            item = item.to(device).long()
            label = label.to(device).float()

            optimizer.zero_grad()
            logits = model(user, item)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * user.size(0)

        epoch_loss = total_loss / len(train_loader.dataset)
        losses.append(epoch_loss)

        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

    return losses

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [26]:
# Hit@K evaluation with negative sampling
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(test_df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()
    all_items_set = set(all_items)
    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in test_df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            neg_pool = list(all_items_set - interacted_items.get(u, set()))
            negatives = np.random.choice(neg_pool, size=num_neg, replace=len(neg_pool) < num_neg)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + list(negatives)
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total

# Train and evaluate MF
num_users = df['user_id'].nunique()
num_items = df['item_id'].nunique()
emb_dim = 32

mf_model = MF(num_users=num_users, num_items=num_items, emb_dim=emb_dim)
mf_losses = train(mf_model, train_df, epochs=5)
mf_hit10 = hit_at_k(mf_model, test_df=test_df, full_df=df, K=10, num_neg=100)
print(f"MF Hit@10: {mf_hit10:.4f}")

Epoch 1, Loss: 0.5115
Epoch 2, Loss: 0.3712
Epoch 3, Loss: 0.3633
Epoch 4, Loss: 0.3591
Epoch 5, Loss: 0.3546
MF Hit@10: 0.5052


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [27]:
# MLP-based recommender
class MLP(nn.Module): 
    def __init__(self, num_users, num_items, emb_dim):  # configure sizes
        super().__init__() 
        self.user_emb = nn.Embedding(num_users, emb_dim)  # user embeddings
        self.item_emb = nn.Embedding(num_items, emb_dim)  # item embeddings
        nn.init.normal_(self.user_emb.weight, std=0.01) 
        nn.init.normal_(self.item_emb.weight, std=0.01)  

        self.mlp = nn.Sequential( 
            nn.Linear(emb_dim * 2, 64),  # concat to hidden
            nn.ReLU(),  # nonlinearity
            nn.Linear(64, 32),  # hidden layer
            nn.ReLU(),  # nonlinearity
            nn.Linear(32, 1),  # output logit
        )

    def forward(self, user, item):  # forward pass
        user_vec = self.user_emb(user.long())  # user vectors
        item_vec = self.item_emb(item.long())  # item vectors
        x = torch.cat([user_vec, item_vec], dim=1)  # concat features
        scores = self.mlp(x).squeeze(1)  # MLP output
        return scores  # logits

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [28]:
# Train and evaluate MLP
mlp_model = MLP(num_users=num_users, num_items=num_items, emb_dim=emb_dim)
mlp_losses = train(mlp_model, train_df, epochs=5)
mlp_hit10 = hit_at_k(mlp_model, test_df=test_df, full_df=df, K=10, num_neg=100)
print(f"MLP Hit@10: {mlp_hit10:.4f}")

Epoch 1, Loss: 0.3978
Epoch 2, Loss: 0.3641
Epoch 3, Loss: 0.3614
Epoch 4, Loss: 0.3601
Epoch 5, Loss: 0.3585
MLP Hit@10: 0.5361


## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

In [29]:
# Comparison and analysis
print(f"\nMF Final Loss: {mf_losses[-1]:.4f}, Hit@10: {mf_hit10:.4f}")
print(f"MLP Final Loss: {mlp_losses[-1]:.4f}, Hit@10: {mlp_hit10:.4f}")

if mlp_hit10 > mf_hit10:
    print(f"\n-> MLP performs better with +{(mlp_hit10 - mf_hit10):.4f} advantage")
else:
    print(f"\n-> MF performs better with +{(mf_hit10 - mlp_hit10):.4f} advantage")


MF Final Loss: 0.3546, Hit@10: 0.5052
MLP Final Loss: 0.3585, Hit@10: 0.5361

-> MLP performs better with +0.0309 advantage


## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [30]:
# Neural Collaborative Filtering (MF + MLP)
class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim): 
        super().__init__() 
        self.mf_user_emb = nn.Embedding(num_users, emb_dim)  # MF user emb
        self.mf_item_emb = nn.Embedding(num_items, emb_dim)  # MF item emb

        self.mlp_user_emb = nn.Embedding(num_users, emb_dim)  # MLP user emb
        self.mlp_item_emb = nn.Embedding(num_items, emb_dim)  # MLP item emb

        nn.init.normal_(self.mf_user_emb.weight, std=0.01)
        nn.init.normal_(self.mf_item_emb.weight, std=0.01)  
        nn.init.normal_(self.mlp_user_emb.weight, std=0.01)  
        nn.init.normal_(self.mlp_item_emb.weight, std=0.01)  

        self.mlp_layers = nn.Sequential(  # MLP tower
            nn.Linear(emb_dim * 2, 64),  # concat to hidden
            nn.ReLU(),  # nonlinearity
            nn.Linear(64, 32),  # hidden layer
            nn.ReLU(),  # nonlinearity
        )

        self.final = nn.Linear(1 + 32, 1)  # fuse MF + MLP

    def forward(self, user, item): 
        user = user.long()  # cast to long
        item = item.long()  # cast to long

        mf_user = self.mf_user_emb(user)  # MF user vec
        mf_item = self.mf_item_emb(item)  # MF item vec
        mf_score = (mf_user * mf_item).sum(dim=1, keepdim=True)  # MF dot

        mlp_user = self.mlp_user_emb(user)  # MLP user vec
        mlp_item = self.mlp_item_emb(item)  # MLP item vec
        mlp_input = torch.cat([mlp_user, mlp_item], dim=1)  # concat feats
        mlp_repr = self.mlp_layers(mlp_input)  # MLP features

        fused = torch.cat([mf_score, mlp_repr], dim=1)  # fuse features
        return self.final(fused).squeeze(1)  # logits


ncf_model = NCF(num_users=num_users, num_items=num_items, emb_dim=emb_dim)  # build model
ncf_losses = train(ncf_model, train_df, epochs=5)  # train model
ncf_hit10 = hit_at_k(ncf_model, test_df=test_df, full_df=df, K=10, num_neg=100)  # eval model
print(f"NCF Hit@10: {ncf_hit10:.4f}")  # report score

comparison = pd.DataFrame(  # compare models
    {  # table data
        'Model': ['MF', 'MLP', 'NCF'],  # names
        'Hit@10': [mf_hit10, mlp_hit10, ncf_hit10],  # scores
    }  # end data
).sort_values('Hit@10', ascending=False)  # best first
print(comparison.to_string(index=False))  # show table

Epoch 1, Loss: 0.3869
Epoch 2, Loss: 0.3274
Epoch 3, Loss: 0.3055
Epoch 4, Loss: 0.2917
Epoch 5, Loss: 0.2792
NCF Hit@10: 0.6818
Model   Hit@10
  NCF 0.681806
  MLP 0.536072
   MF 0.505192
